# Data Extraction Notebook: HAPI FHIR to Raw JSON

1. Requirements & Setup

In [1]:
# !pip install requests tqdm pandas
import requests
import json
import time
import os
from tqdm import tqdm

# Project Constants
BASE_URL = "https://hapi.fhir.org/baseR4"
# We'll target a larger number to see if the gender distribution normalizes
TARGET_TOTAL = 3000 
PAGE_SIZE = 500 # Larger pages reduce the number of requests
RESOURCE_TYPE = "Patient"
OUTPUT_FILE = "raw_fhir_patients.json"

2. The Extraction Function

In [2]:
def extract_all_raw_resources(resource_type="Patient"):
    all_resources = []
    # Using _sort=_id ensures we move through the database linearly
    next_url = f"{BASE_URL}/{resource_type}?_count={PAGE_SIZE}&_sort=_id"
    
    pbar = tqdm(total=TARGET_TOTAL, desc="Global Extraction")

    while next_url and len(all_resources) < TARGET_TOTAL:
        try:
            response = requests.get(next_url, timeout=45)
            if response.status_code == 200:
                bundle = response.json()
                entries = bundle.get('entry', [])
                
                for entry in entries:
                    if len(all_resources) < TARGET_TOTAL:
                        all_resources.append(entry.get('resource'))
                        pbar.update(1)
                
                # Update next_url from the bundle links
                next_url = next((l['url'] for l in bundle.get('link', []) if l['relation'] == 'next'), None)
                
                # The public server appreciates a small breather
                time.sleep(0.5) 
            else:
                print(f"\n[Warning] Server returned status: {response.status_code}")
                break
        except Exception as e:
            print(f"\n[Error] {e}")
            break
            
    pbar.close()
    return all_resources

raw_data = extract_all_raw_resources("Patient")

Global Extraction: 100%|██████████| 3000/3000 [00:13<00:00, 221.34it/s]


3. Data Persistence (Saving the Raw Layer)

In [4]:
def save_raw_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Successfully saved {len(data)} records to {filename}")

save_raw_json(extract_all_raw_resources, OUTPUT_FILE)

TypeError: Object of type function is not JSON serializable

4. Basic Validation

In [ ]:
import pandas as pd

# Preview the first few items
print(f"Sample Resource ID: {raw_resources[0]['id']}")
print(f"Available keys in resource: {raw_resources[0].keys()}")

# Check for missing values in the top-level 'gender' field as a sanity check
genders = [r.get('gender', 'unknown') for r in raw_resources]
print(pd.Series(genders).value_counts())